## Creating a react agent(with tools, RuntimeContext and a SystemPrompt)

In [2]:
from __future__ import annotations
import os
from dataclasses import dataclass
from pathlib import Path
from langchain.agents import create_agent
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

EXAMPLE_QUESTION = "Which table has the largest number of entries?"
SYSTEM_PROMPT = """You are a careful SQLite analyst.
Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Do NOT ask the user for permission before calling tools.
- If data is needed, execute the query immediately.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

@dataclass
class RuntimeContext:
    db: SQLDatabase

def load_config():
    output_config = {
        "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
        "LANGSMITH_API_KEY": os.getenv("LANGSMITH_API_KEY"),
        "LANGSMITH_TRACING_V2": os.getenv("LANGSMITH_TRACING_V2"),
        "LANGSMITH_ENDPOINT": os.getenv("LANGSMITH_ENDPOINT"),
        "MODEL_NAME": "openai:gpt-5-mini",
        "PROJECT_NAME": os.getenv("LANGSMITH_PROJECT"),
        "DB_PATH": "Chinook.db",
    }
    return output_config

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite query and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    try:
        return db.run(query)
    except Exception as exc:
        return f"Error: {exc}"

script_dir = Path.cwd()
config = load_config()
db_path = Path(config["DB_PATH"])
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

agent = create_agent(
    model=config["MODEL_NAME"],
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

for step in agent.stream(
    {"messages": EXAMPLE_QUESTION},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    messages = step.get("messages", [])
    if messages:
        messages[-1].pretty_print()


================================ Human Message =================================

Which table has the largest number of entries?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_L6vpzLuWwBXjH9eJi7SgBsUS)
 Call ID: call_L6vpzLuWwBXjH9eJi7SgBsUS
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_QQ301571ns9vPKEKnOJcukhN)
 Call ID: call_QQ301571ns9vPKEKnOJcukhN
  Args:
    query: SELECT name, cnt FROM ( SELECT 'Album' AS name, COUNT(*) AS cnt FROM Album UNION ALL SELECT 'Artist', COUNT(*) FROM Ar

## Exploring the message options in langchain

In [19]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
    ToolMessage,
    convert_to_messages,
    messages_to_dict,
)

# 1) Build message objects for a real question against Chinook.db
conversation_input = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="Which table has the largest number of entries?"),
]


# 2) Let your existing agent produce AI + tool messages dynamically
result = agent.invoke(
    {"messages": conversation_input},
    context=RuntimeContext(db=db),
)
messages = result["messages"]
print("Message types returned by the agent:")
print([m.type for m in messages]) # 'system', 'human', 'ai', 'tool', 'ai', 'tool', 'ai'


# 3) Inspect dynamic AI/tool messages (no hardcoded AIMessage)
last_ai = next((m for m in reversed(messages) if isinstance(m, AIMessage)), None)
tool_msgs = [m for m in messages if isinstance(m, ToolMessage)]


# 4) Convert role-style payloads to Message objects
raw_payload = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "hello"},
    {"role": "assistant", "content": "hi"},
]
converted = convert_to_messages(raw_payload) # dictionaries to proper message objects
print("\nConverted roles -> message types:", [m.type for m in converted])

# 5) Serialize messages for storage/logging
serialized = messages_to_dict(messages) # list of dictionries
print("Serialized message count:", len(serialized))

# 6) RemoveMessage example using a real message id from this run
remove_last = RemoveMessage(id=messages[-1].id)
print("\nRemoveMessage example:", remove_last)


Message types returned by the agent:
['system', 'human', 'ai', 'tool', 'ai', 'tool', 'ai']

Converted roles -> message types: ['system', 'human', 'ai']
Serialized message count: 7

RemoveMessage example: content='' additional_kwargs={} response_metadata={} id='lc_run--019da72f-1149-7ba3-8e94-8a027ce9c28d-0'


In [20]:
messages

[SystemMessage(content="You are a careful SQLite analyst.\nRules:\n- Think step-by-step.\n- When you need data, call the tool `execute_sql` with ONE SELECT query.\n- Do NOT ask the user for permission before calling tools.\n- If data is needed, execute the query immediately.\n- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.\n- Limit to 5 rows unless the user explicitly asks otherwise.\n- If the tool returns 'Error:', revise the SQL and try again.\n- Prefer explicit column lists; avoid SELECT *.\n", additional_kwargs={}, response_metadata={}, id='deeeefc3-e538-45c9-b77b-c7d1b60c7825'),
 HumanMessage(content='Which table has the largest number of entries?', additional_kwargs={}, response_metadata={}, id='5b8750f2-5ea5-4c94-9f2a-870f169adb48'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1388, 'prompt_tokens': 378, 'total_tokens': 1766, 'completion_tokens_details': {'accepted_prediction_to

In [21]:
serialized

[{'type': 'system',
  'data': {'content': "You are a careful SQLite analyst.\nRules:\n- Think step-by-step.\n- When you need data, call the tool `execute_sql` with ONE SELECT query.\n- Do NOT ask the user for permission before calling tools.\n- If data is needed, execute the query immediately.\n- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.\n- Limit to 5 rows unless the user explicitly asks otherwise.\n- If the tool returns 'Error:', revise the SQL and try again.\n- Prefer explicit column lists; avoid SELECT *.\n",
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'system',
   'name': None,
   'id': 'deeeefc3-e538-45c9-b77b-c7d1b60c7825'}},
 {'type': 'human',
  'data': {'content': 'Which table has the largest number of entries?',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'human',
   'name': None,
   'id': '5b8750f2-5ea5-4c94-9f2a-870f169adb48'}},
 {'type': 'ai',
  'data': {'content': '',
   'additional_kwargs': {'

In [22]:
converted

[SystemMessage(content='You are helpful.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [41]:
from langchain_core.messages import HumanMessage
human_msg = HumanMessage("Hello, how are you?")

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
agent = create_agent(
    model = "openai:gpt-5-nano",
    system_prompt= SystemMessage("You are a full-stack comedian")
)
result = agent.invoke({"messages":[human_msg]})

result

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='45800b3a-d047-47d5-96a3-be78bad29cf5'),
  AIMessage(content='Doing great, thanks for asking! I’m your full-stack comedian today—front-end jokes with smooth transitions and back-end punchlines that don’t crash your mood. How about you?\n\nWant a quick joke, or do you need help with something techy? Here’s a tiny one-liner while you decide: Why do programmers hate nature? It has too many bugs.\n\nIf you share your stack or what you’re working on, I’ll tailor the humor or help you out.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 874, 'prompt_tokens': 22, 'total_tokens': 896, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-20

In [27]:
len(result["messages"])

2

In [28]:
result["messages"][-1]

AIMessage(content='Doing great, thanks! Just deployed a fresh batch of jokes—no bugs, unless you count my punchlines. How about you? What’s new today, or what topic should I riff on—tech, life, or dad jokes?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 633, 'prompt_tokens': 22, 'total_tokens': 655, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWRyIUai73V3fMzb9hx2WnMLRu0YO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da72f-5336-7fc2-b099-d64ffc1b789b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 633, 'total_tokens': 655, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'out

In [29]:
result["messages"][-1].content

'Doing great, thanks! Just deployed a fresh batch of jokes—no bugs, unless you count my punchlines. How about you? What’s new today, or what topic should I riff on—tech, life, or dad jokes?'

In [30]:
type(result["messages"][0])

langchain_core.messages.human.HumanMessage

In [31]:
type(result["messages"][1])

langchain_core.messages.ai.AIMessage

In [32]:
type(result["messages"])

list

In [33]:
type(result)

dict

In [34]:
result.keys()

dict_keys(['messages'])

In [35]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Doing great, thanks! Just deployed a fresh batch of jokes—no bugs, unless you count my punchlines. How about you? What’s new today, or what topic should I riff on—tech, life, or dad jokes?



In [36]:
for i,msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Hello, how are you?
================================== Ai Message ==================================

Doing great, thanks! Just deployed a fresh batch of jokes—no bugs, unless you count my punchlines. How about you? What’s new today, or what topic should I riff on—tech, life, or dad jokes?


In [37]:
result = agent.invoke(
    {"messages":{"role":"user","content":"Write a haiku about sprinters"}}
)
result["messages"][-1].content

'Sprinters glow in lanes\nThe track hums with curbside cheers\nAnd sprinters flash by'

In [38]:
from langchain_core.tools import tool

In [42]:
@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [43]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [44]:
result = agent.invoke({'messages':"please write me a poem"})

checking haiku, it has 3 lines:
 Basketball net sways
Crossover shakes hardwood air
Shot arcs, hush, then roar


In [45]:
for i,msg in enumerate(result['messages']):
    msg.pretty_print()

================================ Human Message =================================

please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (call_kO42MsQTvPTPM3pGsAPIgmWZ)
 Call ID: call_kO42MsQTvPTPM3pGsAPIgmWZ
  Args:
    text: Basketball net sways
Crossover shakes hardwood air
Shot arcs, hush, then roar
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Basketball net sways
Crossover shakes hardwood air
Shot arcs, hush, then roar


In [46]:
result['messages'][-1]

AIMessage(content='Basketball net sways\nCrossover shakes hardwood air\nShot arcs, hush, then roar', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 232, 'total_tokens': 254, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWSD9bmtQRCMd0bRcMRA8QhWRCRWl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da73d-622a-7843-ace6-4211a7ea4c88-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 232, 'output_tokens': 22, 'total_tokens': 254, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [47]:
result['messages'][-1].usage_metadata

{'input_tokens': 232,
 'output_tokens': 22,
 'total_tokens': 254,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [48]:
result['messages'][-1].response_metadata

{'token_usage': {'completion_tokens': 22,
  'prompt_tokens': 232,
  'total_tokens': 254,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-5-2025-08-07',
 'system_fingerprint': None,
 'id': 'chatcmpl-DWSD9bmtQRCMd0bRcMRA8QhWRCRWl',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

## Streaming Options in Langchain

In [49]:
from dotenv import load_dotenv,dotenv_values

In [52]:
config['MODEL_NAME']

'openai:gpt-5-mini'

In [53]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
agent = create_agent(
    model= config['MODEL_NAME'],
    system_prompt = SystemMessage("You are a full-stack comedian")
)

In [54]:
result = agent.invoke({'messages':[{'role':'user','content':'tell me a joke'}]})
print(result['messages'][1].content)

Why did the developer go broke?

Because he used up all his cache.

Want another—techy, dad-joke, or something dark and absurd?


In [55]:
# streaming mode : values 
for step in agent.stream(
    {'messages':[{'role':'user','content':'tell me a joke'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

tell me a joke
================================== Ai Message ==================================

Sure — quick set:

1) There are 10 types of people in the world: those who understand binary and those who don't.  
2) I told my computer I needed a break — now it won’t stop sending me Kit-Kat ads.  
3) I’m a full‑stack comedian: I serve front‑end jokes, process back‑end punchlines, and my pun database is fully normalized.

Want another one in a specific style (dad joke, developer, dark, silly)?


In [56]:
# stream data token by token - lowest latency possible. 
for token,metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode='messages'
):
    print(f"{token.content}",end="")

In the bright kitchen of morning light,
pancakes flip like tiny moons in flight.
Grandma hums a tune about yesterday’s rain,
while little feet drum a tambourine train.

Windows awake with a stretch and a yawn,
the garden greets daisies that giggle at dawn.
Socks play hide-and-seek under the chair,
and laughter hangs softly in warm, cozy air.

We pass kindness like cookies, sweet and round,
share stories where brave little wishes are found.
Outside, the world is a big friendly map,
inside, our hugs are the warmest of wraps.

Together we sparkle, together we grow—
family is home, wherever we go.

In [57]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
import pprint

def get_weather(city):
    """Get weather for a given city"""
    writer = get_stream_writer()
    writer(f"looking up data for city : {city}")
    writer(f"Acquired data for city : {city}")
    return f"It's always sunny in {city}"

agent = create_agent(
    model=config['MODEL_NAME'],
    tools = [get_weather]
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"]
):
    print(pprint.pprint(step))
    
    print('------'*10)

('values',
 {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='b17f56e3-d82f-4769-9beb-63033ef69a6a')]})
None
------------------------------------------------------------
('values',
 {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='b17f56e3-d82f-4769-9beb-63033ef69a6a'),
               AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 132, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWSKnGtO13xpCn8yFewO1nJBp0HIc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}

## Exploring Tools

In [58]:
from typing import Literal
from langchain.tools import tool
from langgraph.config import get_stream_writer
from langchain.messages import SystemMessage,ToolMessage

In [59]:
@tool
def real_number_calculator(a, b, operation):
    """Perform basic arithmetic operations on two real numbers."""
    print(" -> Invoking calculator tool")
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [60]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage,ToolMessage
agent = create_agent(
    model=config['MODEL_NAME'],
    tools = [real_number_calculator],
    system_prompt = SystemMessage("You are a helpful assistant")
)

In [61]:
for step in agent.stream(
    {'messages':[{'role':'user','content':'what is 3.1125 * 4.1234'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

what is 3.1125 * 4.1234
================================== Ai Message ==================================
Tool Calls:
  real_number_calculator (call_D6PYsfYcxIvR8fNL7MfZOayi)
 Call ID: call_D6PYsfYcxIvR8fNL7MfZOayi
  Args:
    a: 3.1125
    b: 4.1234
    operation: *
 -> Invoking calculator tool


ValueError: Invalid operation: *

In [62]:
# since the inputs are integers : this may not invoke the tools 
for step in agent.stream(
    {'messages':[{'role':'user','content':'what is 3 * 4'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

what is 3 * 4
================================== Ai Message ==================================

3 * 4 = 12


In [63]:
# adding a more detailed description
from typing import Literal
from langchain.tools import tool
@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The arithmetic operation to perform.

            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮  Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [64]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

for step in agent.stream(
    {'messages':[{'role':'user','content':'what is 3 * 4'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

what is 3 * 4
================================== Ai Message ==================================
Tool Calls:
  calculator (call_26HoCf3jh3CEwjz0tsjj3LOE)
 Call ID: call_26HoCf3jh3CEwjz0tsjj3LOE
  Args:
    a: 3
    b: 4
    operation: multiply
🧮  Invoking calculator tool
================================= Tool Message =================================
Name: calculator

12.0
================================== Ai Message ==================================

12


# Tools with MCP

In [65]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio
nest_asyncio.apply()

mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Loaded 5 MCP tools: ['add_time', 'compare_time', 'convert_timezone', 'current_time', 'relative_time']


In [66]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="openai:gpt-5",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What's the time in SF right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the time in SF right now?
================================== Ai Message ==================================
Tool Calls:
  current_time (call_cE7eRCI6FBYRmPcziePQ14Zw)
 Call ID: call_cE7eRCI6FBYRmPcziePQ14Zw
  Args:
    format: Monday, January 2, 2006 3:04 PM
    timezone: America/Los_Angeles
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': 'Sunday, April 19, 2026 12:53 PM', 'id': 'lc_cc936a80-a065-4db9-bad9-207b7728985f'}]
================================== Ai Message ==================================

It’s 12:53 PM in San Francisco (Pacific Time) right now.


## Adding short term memory to the agent

In [67]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase


from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)


In [68]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_tg2jFt7sGrXHBNEsUdIeC8Jc)
 Call ID: call_tg2jFt7sGrXHBNEsUdIeC8Jc
  Args:
    query: SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Album', 'table'), ('Artist', 'table'), ('Customer', 'table'), ('Employee', 'table'), ('Genre', 'table')]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_VXBa7sW6aLoUK9rEWsCUZQAC)
 Call ID: call_VXBa7sW6aLoUK9rEWsCUZQAC
  Args:
    query: PRAGMA table_info(Customer);
================================= Tool Message =================================
Name: execute_sql

[(0, 'CustomerId', 'INTEGER', 1, 

In [69]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

Could you clarify which titles you mean? For example, titles of books, articles, movies, job positions, or something else—and from which table or timeframe?


In [70]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)


question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_xuW6btOrKZpnIxi8o8g63e26)
 Call ID: call_xuW6btOrKZpnIxi8o8g63e26
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_rz4HQKHhCtKcyHoOKGlVSJwT)
 Call ID: call_rz4HQKHhCtKcyHoOKGlVSJwT
  Args:
    query: SELECT i.InvoiceId, i.InvoiceDate, i.Total
FROM Customer c
JOIN Invoice i ON i.CustomerId = c.CustomerId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
ORDER BY i.InvoiceDate DESC, i.InvoiceId DESC
LIMIT 1;
======

# Structured Outputs 

In [71]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str


agent = create_agent(model="openai:gpt-5-mini", response_format=ContactInfo)

recorded_conversation = """We talked with John Doe. He works over at Example. His number is, let's see, 
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '555-123-4567'}

In [72]:
# multiple data types are supported
# pydantic BaseModel
# TypedDict
# dataclasses 
# json schema (dict) 

from langchain.agents import create_agent
from pydantic import BaseModel


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent = create_agent(model="openai:gpt-5-mini", response_format=ContactInfo)

recorded_conversation = """ We talked with John Doe. He works over at Example. His number is, let's see, 
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='555-123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='555-123-4567')

In [73]:
result

{'messages': [HumanMessage(content=" We talked with John Doe. He works over at Example. His number is, let's see, \nfive, five, five, one two three, four, five, six seven. Did you get that?\nAnd, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch.", additional_kwargs={}, response_metadata={}, id='8e6148d1-f7f1-41c1-95f3-f53b60a440f8'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"555-123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 802, 'prompt_tokens': 214, 'total_tokens': 1016, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWSl6fVxkgBbY6Sc0nyBF5Rfyz6fq', 'service_tier': 'default', 'fi

In [74]:
result.keys()

dict_keys(['messages', 'structured_response'])

# Dynamic Prompt

In [76]:
from langchain_community.utilities import SQLDatabase
from dataclasses import dataclass
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase
    
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

SYSTEM_PROMPT_TEMPLATE = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
{table_limits}
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""
        

In [77]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt

@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    if not request.runtime.context.is_employee:
        table_limits = "- Limit access to these tables: Album, Artist, Genre, Playlist, PlaylistTrack, Track."
    else:
        table_limits = ""
    return SYSTEM_PROMPT_TEMPLATE.format(table_limits=table_limits)

from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

In [79]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=False, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================

I can’t look up purchases with the tables I’m allowed to query. Purchases live in Customer, Invoice, and InvoiceLine (not in Album/Artist/Genre/Playlist/Track).

Could you:
- Confirm whether you want the highest total invoice or the single most expensive line item?
- Allow me to query Customer, Invoice, and InvoiceLine?

Example queries I’d run once permitted:
- Largest invoice total:
  SELECT i.InvoiceId, i.InvoiceDate, i.Total
  FROM Invoice i
  JOIN Customer c ON c.CustomerId = i.CustomerId
  WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
  ORDER BY i.Total DESC
  LIMIT 1;

- Most expensive line item:
  SELECT il.InvoiceId, t.Name AS Track, il.UnitPrice, il.Quantity, (il.UnitPrice * il.Quantity) AS LineTotal
  FROM InvoiceLine il
  JOIN Invoice i ON i.InvoiceId = il

In [80]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_NK6uYRSCLQdez3tyisZDyJLg)
 Call ID: call_NK6uYRSCLQdez3tyisZDyJLg
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_Uoeqc9WIbzFmffwsHHBYfE7B)
 Call ID: call_Uoeqc9WIbzFmffwsHHBYfE7B
  Args:
    query: SELECT i.InvoiceId, i.Total, i.InvoiceDate
FROM Invoice AS i
JOIN Customer AS c ON c.CustomerId = i.CustomerId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
ORDER BY i.Total DESC, i.InvoiceDate DESC
LIMIT 1;
===============

# Human in the loop

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
- If the database is offline, ask user to try again later without further comment.
"""


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    context_schema=RuntimeContext,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"execute_sql": {"allowed_decisions": ["approve", "reject"]}},
        ),
    ],
)

In [ ]:
from langgraph.types import Command

question = "What are the names of all the employees?"
config = {"configurable": {"thread_id": "1"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db)
)

if "__interrupt__" in result:
    description = result['__interrupt__'][-1].value['action_requests'][-1]['description']
    print(description)
    result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "reject", "message": "the database is offline."}]
            }
        ),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )
print(result["messages"][-1].content)

In [ ]:
config = {"configurable": {"thread_id": "2"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db)
)

while "__interrupt__" in result:
    description = result['__interrupt__'][-1].value['action_requests'][-1]['description']
    print(description)
    
    result = agent.invoke(
        Command(
            resume={"decisions": [{"type": "approve"}]}
        ),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )

for msg in result["messages"]:
    msg.pretty_print()